In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import time
import itertools

from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier

from scipy.stats import uniform

In [2]:
import pickle

with open('data/processed/tree_models_data.pkl', 'rb') as f:
    tree_data = pickle.load(f)

X_train_proc = tree_data['X_train_proc']
X_val_proc = tree_data['X_val_proc']
X_test_proc = tree_data['X_test_proc']
y_train = tree_data['y_train']
y_val = tree_data['y_val']
y_test = tree_data['y_test']

tree_results = tree_data['tree_results']

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/tree_models_data.pkl'

#### Architecture comparison

In [ ]:
# 1. Hidden layer
hidden_layers = [(50,), (100,), (200,), (50, 50), (100, 50), (100, 100)]

# 2. Activation functions
activations = ['relu', 'tanh']

# 3. Optimizers
solvers = ['adam', 'sgd']

# 4. Learning rates
learning_rates = [0.001, 0.01, 0.1]

# Storing results
results = []

# ReLu and Adam
print("Testing different neural network architectures...")
for hidden in hidden_layers:
    start_time = time.time()
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=300,
        random_state=42
    )
    
    mlp.fit(X_train_proc, y_train)
    
    train_acc = mlp.score(X_train_proc, y_train)
    val_acc = mlp.score(X_val_proc, y_val)
    
    results.append({
        'hidden_layers': str(hidden),
        'activation': 'relu',
        'solver': 'adam',
        'learning_rate': 0.001,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'time': time.time() - start_time
    })
    
    print(f"Architecture {hidden}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

#### Activation Function Comparison

In [ ]:
best_arch = max(results, key=lambda x: x['val_acc'])['hidden_layers']
best_arch = eval(best_arch)  # Convert string back to tuple

print(f"\nTesting different activation functions with {best_arch} architecture...")
for activation in activations:
    start_time = time.time()
    mlp = MLPClassifier(
        hidden_layer_sizes=best_arch,
        activation=activation,
        solver='adam',
        learning_rate_init=0.001,
        max_iter=300,
        random_state=42
    )
    
    mlp.fit(X_train_proc, y_train)
    
    train_acc = mlp.score(X_train_proc, y_train)
    val_acc = mlp.score(X_val_proc, y_val)
    
    results.append({
        'hidden_layers': str(best_arch),
        'activation': activation,
        'solver': 'adam',
        'learning_rate': 0.001,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'time': time.time() - start_time
    })
    
    print(f"Activation {activation}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

#### Optimizer and learning rate experiments

In [ ]:
best_activation = max([r for r in results if r['hidden_layers'] == str(best_arch)], 
                      key=lambda x: x['val_acc'])['activation']

print(f"\nTesting different optimizers and learning rates with {best_arch} architecture and {best_activation} activation...")
for solver, lr in itertools.product(solvers, learning_rates):
    start_time = time.time()
    mlp = MLPClassifier(
        hidden_layer_sizes=best_arch,
        activation=best_activation,
        solver=solver,
        learning_rate_init=lr,
        max_iter=300,
        random_state=42
    )
    
    mlp.fit(X_train_proc, y_train)
    
    train_acc = mlp.score(X_train_proc, y_train)
    val_acc = mlp.score(X_val_proc, y_val)
    
    results.append({
        'hidden_layers': str(best_arch),
        'activation': best_activation,
        'solver': solver,
        'learning_rate': lr,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'time': time.time() - start_time
    })
    
    print(f"Solver {solver}, LR {lr}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

#### Random initialization stability test

In [ ]:
# Try different random initializations with best parameters
best_result = max(results, key=lambda x: x['val_acc'])
best_hidden = eval(best_result['hidden_layers'])
best_activation = best_result['activation']
best_solver = best_result['solver']
best_lr = best_result['learning_rate']

print(f"\nTesting different random initializations with best parameters...")
random_inits = []
for seed in range(5):
    start_time = time.time()
    mlp = MLPClassifier(
        hidden_layer_sizes=best_hidden,
        activation=best_activation,
        solver=best_solver,
        learning_rate_init=best_lr,
        max_iter=300,
        random_state=seed
    )
    
    mlp.fit(X_train_proc, y_train)
    
    train_acc = mlp.score(X_train_proc, y_train)
    val_acc = mlp.score(X_val_proc, y_val)
    
    random_inits.append({
        'seed': seed,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'time': time.time() - start_time
    })
    
    print(f"Seed {seed}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

#### Results

In [ ]:
# Convert results to DataFrame for easier analysis
results_df = pd.DataFrame(results)

# Visualization of hyperparameter effects
plt.figure(figsize=(15, 12))

# 1. Plot network architecture comparison
plt.subplot(2, 2, 1)
arch_data = results_df[results_df['activation'] == 'relu'][results_df['solver'] == 'adam'][results_df['learning_rate'] == 0.001]
plt.plot(arch_data['hidden_layers'], arch_data['train_acc'], 'bo-', label='Train Accuracy')
plt.plot(arch_data['hidden_layers'], arch_data['val_acc'], 'ro-', label='Validation Accuracy')
plt.title('Neural Network Architecture Comparison')
plt.xlabel('Hidden Layer Architecture')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)

# 2. Plot activation function comparison
plt.subplot(2, 2, 2)
act_data = results_df[results_df['hidden_layers'] == str(best_arch)][results_df['solver'] == 'adam'][results_df['learning_rate'] == 0.001]
plt.bar(act_data['activation'].astype(str) + ' (Train)', act_data['train_acc'], color='blue', alpha=0.7)
plt.bar(act_data['activation'].astype(str) + ' (Val)', act_data['val_acc'], color='red', alpha=0.7)
plt.title('Activation Function Comparison')
plt.xlabel('Activation Function')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.grid(True)

# 3. Plot optimizer and learning rate comparison
plt.subplot(2, 2, 3)
opt_data = results_df[results_df['hidden_layers'] == str(best_arch)][results_df['activation'] == best_activation]
for solver in solvers:
    solver_data = opt_data[opt_data['solver'] == solver]
    plt.plot(solver_data['learning_rate'], solver_data['val_acc'], 'o-', label=f'{solver}')
plt.title('Optimizer and Learning Rate Comparison')
plt.xlabel('Learning Rate')
plt.ylabel('Validation Accuracy')
plt.xscale('log')
plt.legend()
plt.grid(True)

# 4. Plot random initialization comparison
plt.subplot(2, 2, 4)
ri_df = pd.DataFrame(random_inits)
plt.bar(ri_df['seed'].astype(str) + ' (Train)', ri_df['train_acc'], color='blue', alpha=0.7)
plt.bar(ri_df['seed'].astype(str) + ' (Val)', ri_df['val_acc'], color='red', alpha=0.7)
plt.title('Random Initialization Comparison')
plt.xlabel('Random Seed')
plt.ylabel('Accuracy')
plt.grid(True)

plt.tight_layout()
plt.savefig('nn_hyperparameter_comparison.png', dpi=300)
plt.show()

#### Final tuning with Random Search

In [ ]:
print("\n\nProper Hyperparameter Tuning using RandomizedSearchCV...")

param_distributions = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50), (100, 100)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'learning_rate_init': uniform(0.0001, 0.01),
    'alpha': uniform(0.0001, 0.001),  # L2 regularization
    'batch_size': [32, 64, 128, 'auto']
}

# Neural network tuning timing
nn_tuning_start = time.time()

# Random Search
random_search = RandomizedSearchCV(
    estimator=MLPClassifier(max_iter=300, random_state=42),
    param_distributions=param_distributions,
    n_iter=20,  # Try 20 random combinations
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_proc, y_train)

nn_tuning_time = time.time() - nn_tuning_start

# Best parameters and best model
best_params = random_search.best_params_
best_score = random_search.best_score_
print(f"Best parameters: {best_params}")
print(f"Best validation score: {best_score:.4f}")

nn_training_start = time.time()
best_mlp = MLPClassifier(
    hidden_layer_sizes=best_params['hidden_layer_sizes'],
    activation=best_params['activation'],
    solver=best_params['solver'],
    learning_rate_init=best_params['learning_rate_init'],
    alpha=best_params['alpha'],
    batch_size=best_params['batch_size'],
    max_iter=300,
    random_state=42
)
best_mlp.fit(X_train_proc, y_train)
nn_training_time = time.time() - nn_training_start

nn_test_acc = best_mlp.score(X_test_proc, y_test)
print(f"Neural Network Test Accuracy: {nn_test_acc:.4f}")
print(f"Neural Network Tuning Time: {nn_tuning_time:.2f} seconds")
print(f"Neural Network Training Time: {nn_training_time:.2f} seconds")

In [ ]:
nn_data = {
    'X_train_proc': X_train_proc,
    'X_test_proc': X_test_proc,
    'y_test': y_test,
    'nn_results': results,  # Your NN results DataFrame
    'best_nn_model': best_model,
    'tree_results': tree_results  # Carry forward tree results
}

with open('data/processed/nn_models_data.pkl', 'wb') as f:
    pickle.dump(nn_data, f)